# Μάθημα 13 - Μνήμη Πράκτορα


## Setup

Αυτό το σημειωματάριο δείχνει πώς να δημιουργήσετε έναν πράκτορα κρατήσεων ταξιδιών με **επίμονη μνήμη** χρησιμοποιώντας το **Microsoft Agent Framework** (MAF).

Θα μάθετε πώς διαφορετικοί τύποι μνήμης πράκτορα — εργασίας, βραχυπρόθεσμη και μακροπρόθεσμη — επηρεάζουν το πώς ένας πράκτορας διατηρεί και χρησιμοποιεί πληροφορίες σε συνομιλίες.

**Προαπαιτούμενα:**
- Ένα έργο Azure AI Foundry με αναπτυγμένο μοντέλο συνομιλίας (π.χ. `gpt-4o-mini`).
- Έχετε συνδεθεί με το Azure CLI — εκτελέστε `az login` στο τερματικό σας.
- `AZURE_AI_PROJECT_ENDPOINT` — το endpoint του έργου Azure AI Foundry.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — το όνομα του αναπτυγμένου μοντέλου σας.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Τύποι Μνήμης Αντιπροσώπου

Οι πράκτορες τεχνητής νοημοσύνης μπορούν να αξιοποιήσουν διάφορα είδη μνήμης, καθένα από τα οποία εξυπηρετεί διαφορετικό σκοπό:

### Εργασιακή Μνήμη
Το ίδιο το νήμα της συζήτησης — τα μηνύματα που ανταλλάσσονται σε μια ενιαία συνεδρία. Ο αντιπρόσωπος μπορεί να ανατρέχει σε προηγούμενα μηνύματα στο ίδιο νήμα για να διατηρήσει τη συνοχή. Στο MAF αυτό δημιουργείται μέσω της **`agent.create_session()`**, η οποία επιστρέφει ένα `AgentSession`.

### Βραχυπρόθεσμη Μνήμη
Πληροφορίες που διαρκούν για τη διάρκεια μιας εργασίας ή συνεδρίας, αλλά δεν αποθηκεύονται μόνιμα. Για παράδειγμα, ο πράκτορας μπορεί να συσσωρεύσει γεγονότα κατά τη διάρκεια μιας πολύστροφης συνομιλίας σχεδιασμού και να τα χρησιμοποιήσει για να παράγει ένα τελικό πρόγραμμα.

### Μακροπρόθεσμη Μνήμη
Προτιμήσεις και γεγονότα που διαρκούν **μέσω των συνεδριών**. Ένας επιστρεφόμενος χρήστης δεν πρέπει να επαναλαμβάνει τους διατροφικούς περιορισμούς ή το στυλ ταξιδιού του. Η μακροπρόθεσμη μνήμη συνήθως υποστηρίζεται από έναν εξωτερικό αποθηκευτικό χώρο — μια βάση δεδομένων, αρχείο ή ευρετήριο διανυσμάτων — και παρέχεται στον αντιπρόσωπο μέσω εργαλείων.


## Εργαζόμενη Μνήμη με Συνεδρίες

Η απλούστερη μορφή μνήμης είναι η συνεδρία συνομιλίας. Όταν περνάτε την ίδια συνεδρία (δημιουργημένη μέσω `agent.create_session()`) σε διαδοχικές κλήσεις `agent.run()`, ο πράκτορας βλέπει ολόκληρο το ιστορικό αυτής της συνομιλίας και μπορεί να θυμηθεί προηγούμενες λεπτομέρειες.

Ας δημιουργήσουμε έναν ταξιδιωτικό πράκτορα και να παρουσιάσουμε την εργαζόμενη μνήμη.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Ο πράκτορας ανέκτησε σωστά τον προϋπολογισμό επειδή και τα δύο μηνύματα μοιράζονται την ίδια συνεδρία. Αυτή είναι η **εργαζόμενη μνήμη** — υπάρχει μόνο για τη διάρκεια της συνεδρίας.

### Τι συμβαίνει με μια νέα συζήτηση;

Αν δημιουργήσουμε μια **νέα** συνεδρία, ο πράκτορας δεν έχει μνήμη της προηγούμενης συνομιλίας:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Πρότυπο Μακροπρόθεσμης Μνήμης

Για να θυμόμαστε τις προτιμήσεις του χρήστη **σε διάφορες συνεδρίες**, χρειαζόμαστε μια επίμονη αποθήκευση που βρίσκεται έξω από το νήμα της συζήτησης. Ο πράκτορας έχει πρόσβαση σε αυτήν την αποθήκη μέσω **εργαλείων** — συναρτήσεις που μπορεί να καλέσει για να αποθηκεύσει και να ανακτήσει πληροφορίες.

Παρακάτω υλοποιούμε μια απλή αποθήκη προτιμήσεων στη μνήμη (στην παραγωγή θα τη στηρίζατε με μια βάση δεδομένων ή ευρετήριο διανυσμάτων) και την εκθέτουμε ως εργαλεία που μπορεί να χρησιμοποιήσει ο πράκτορας.

### Αρχιτεκτονική
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Σενάριο 1 — Ο χρήστης που επισκέπτεται για πρώτη φορά κλείνει ένα ταξίδι επετείου

Η Σάρα επισκέπτεται για πρώτη φορά. Ο πράκτορας θα πρέπει να αποθηκεύει τις προτιμήσεις της μέσω των εργαλείων και να τις χρησιμοποιεί για να προτείνει ξενοδοχεία.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Σενάριο 2 — Η Σάρα επιστρέφει εβδομάδες αργότερα

Η Σάρα ξεκινάει ένα **εντελώς νέο νήμα** (μιμούμενη μια νέα συνεδρία). Η προσωρινή μνήμη είναι κενή, αλλά το αποθηκευτικό της προτιμήσεων μακροπρόθεσμα εξακολουθεί να έχει τις πληροφορίες της. Ο πράκτορας θα πρέπει να τις ανακτήσει και να τις χρησιμοποιήσει για να προσωποποιήσει τις προτάσεις.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Περίληψη

Σε αυτό το μάθημα μάθατε τρεις τύπους μνήμης πράκτορα και πώς να τους υλοποιήσετε με το Microsoft Agent Framework:

| Τύπος Μνήμης | Μηχανισμός MAF | Διάρκεια |
|---|---|---|
| **Εργασίας** | `agent.create_session()` | Μοναδική συνομιλία |
| **Βραχυπρόθεσμη** | Συσσωρευμένο πλαίσιο μέσα σε ένα νήμα | Μοναδική εργασία / συνεδρία |
| **Μακροπρόθεσμη** | Εξωτερικό αποθετήριο προσβάσιμο μέσω των συναρτήσεων `@tool` | Μεταξύ συνεδριών |

### Βασικά Συμπεράσματα
1. **`agent.create_session()`** παρέχει μνήμη εργασίας — ο πράκτορας βλέπει την πλήρη ιστορία συνομιλίας μέσα σε μια συνεδρία.
2. **Οι νέες συνεδρίες χάνουν το πλαίσιο** — χωρίς μακροπρόθεσμη μνήμη ο πράκτορας δεν μπορεί να ανακαλέσει παλιές συνομιλίες.
3. **Οι συναρτήσεις `@tool` γεφυρώνουν το κενό** — επιτρέπουν στον πράκτορα να αποθηκεύει και να ανακτά πληροφορίες από ένα επίμονο αποθετήριο.
4. **Η εξατομίκευση βελτιώνεται με τον χρόνο** — όσο περισσότερες προτιμήσεις αποθηκεύονται, τόσο καλύτερες είναι οι προτάσεις του πράκτορα.

### Εφαρμογές στην Πράξη
- **Εξυπηρέτηση Πελατών**: Να θυμάται ιστορικό και προτιμήσεις πελατών
- **Προσωπικοί Βοηθοί**: Να διατηρεί πλαίσιο μέσα σε μέρες ή εβδομάδες
- **Υγεία**: Παρακολούθηση πληροφοριών και προτιμήσεων ασθενών
- **Ηλεκτρονικό Εμπόριο**: Εξατομικευμένες αγορές βάσει ιστορικού

### Επόμενα Βήματα
- Αντικαταστήστε το dictionary στη μνήμη με βάση δεδομένων ή vector store (π.χ. Azure AI Search)
- Προσθέστε λήξη μνήμης για χρονικά ευαίσθητες πληροφορίες
- Δημιουργήστε πολυπράκτορα συστήματα με κοινή μνήμη
- Εξερευνήστε το σημειωματάριο Cognee για μνήμη βασισμένη σε γράφημα γνώσης


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:  
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης AI [Co-op Translator](https://github.com/Azure/co-op-translator). Παρόλο που προσπαθούμε για ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις μπορεί να περιέχουν σφάλματα ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική μετάφραση από άνθρωπο. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
